In [1]:
import pickle
import pandas as pd
import numpy as np

In [2]:
blind = {33, 35, 36, 38, 39, 41, 42, 43, 53}
ctrlA = {3, 4, 5, 6, 7, 8, 9, 10, 11, 27}
ctrlAV = {12, 13, 14, 15, 16, 17, 18, 19, 22, 32}

participant_to_group = {}

for p in blind:
    participant_to_group[p] = "blind"
for p in ctrlA:
    participant_to_group[p] = "ctrlA"
for p in ctrlAV:
    participant_to_group[p] = "ctrlAV"

run_start_indices = [0, 267, 492, 812, 1137, 1373]

In [3]:
def convert_nested_dict_to_df(data_dict, participant_to_group):
    df_list = []
    
    for participant, roi_dict in data_dict.items():
        for roi, model_dict in roi_dict.items():

            temp = (
                pd.DataFrame(model_dict)
                .reset_index()
                .melt(id_vars="index", var_name="model", value_name="value")
            )

            temp["run"] = np.searchsorted(run_start_indices, temp["index"], side="right")

            temp["participant"] = participant
            temp["roi"] = roi
            temp["roi_type"] = temp["roi"].apply(lambda x: "language" if isinstance(x, (int, float)) else "visual")
            temp["participant_group"] = participant_to_group[participant]

            df_list.append(temp)
    return pd.concat(df_list, ignore_index=True)

In [19]:
#for the per run

import pandas as pd
import numpy as np

def convert_nested_dict_to_df(data_dict, participant_to_group, run_start_indices=[0, 267, 492, 812, 1137, 1373]):
    df_list = []
    
    for participant, roi_dict in data_dict.items():
        for roi, model_dict in roi_dict.items():
            # Ensure all values are arrays/lists
            safe_model_dict = {}
            for model_name, value in model_dict.items():
                if np.isscalar(value):
                    safe_model_dict[model_name] = [value]
                else:
                    safe_model_dict[model_name] = value

            # Create DataFrame directly from values
            temp = pd.DataFrame(safe_model_dict).melt(
                var_name="model",
                value_name="value"
            )

            # Compute run index if needed
            temp["run"] = np.searchsorted(run_start_indices, temp.index, side="right")

            # Add metadata
            temp["participant"] = participant
            temp["roi"] = roi
            temp["roi_type"] = "language" if isinstance(roi, (int, float)) else "visual"
            temp["participant_group"] = participant_to_group[participant]

            df_list.append(temp)
    
    return pd.concat(df_list, ignore_index=True)

In [20]:
with open("results_perrun.pkl", "rb") as f:
    data = pickle.load(f)

df = convert_nested_dict_to_df(data, participant_to_group)

# with open("january/qwen-omni/results.pkl", "rb") as f:
#     data = pickle.load(f)

# df2 = convert_nested_dict_to_df(data, participant_to_group)

# with open("january/other/results.pkl", "rb") as f:
#     data = pickle.load(f)

# df3 = convert_nested_dict_to_df(data, participant_to_group)

In [6]:
df = pd.concat([df1, df2, df3], ignore_index=True)

In [21]:
df

,model,value,run,participant,roi,roi_type,participant_group
0,CLIPmultilingualmulti_layers11-15,0.063751,1,33,2,language,blind
1,qwen-omni_audiovideotext_layers16-20_conv,0.068916,1,33,2,language,blind
2,fasttext_conv,0.063741,1,33,2,language,blind
3,fasttext-grounded_conv,0.075487,1,33,2,language,blind
4,qwen-text_layers16-20_conv,0.067466,1,33,2,language,blind
...,...,...,...,...,...,...,...
865,qwen-omni_audiovideotext_layers16-20_conv,0.029002,1,32,visual,visual,ctrlAV
866,fasttext_conv,0.021409,1,32,visual,visual,ctrlAV
867,fasttext-grounded_conv,0.025610,1,32,visual,visual,ctrlAV
868,qwen-text_layers16-20_conv,0.032187,1,32,visual,visual,ctrlAV


In [6]:
with open("january/semantic/results_semantic.pkl", "rb") as f:
    semanticresults = pickle.load(f)

print(semanticresults)

FileNotFoundError: [Errno 2] No such file or directory: 'january/semantic/results_semantic.pkl'

In [7]:
semantic_models = set(m for _, m in semanticresults.keys())
df_models = set(df["model"].unique())

print("Semantic-only models:", semantic_models - df_models)
print("DF-only models:", df_models - semantic_models)


NameError: name 'semanticresults' is not defined

In [33]:
MODEL_MAP = {
    "highlevel_word2vec_72pcs_conv": "word2vec",
    "fasttext_conv": "fasttext",
    # identity mappings are optional but can be explicit:
    "qwen-text_layers28-32_conv": "qwen-text_layers28-32_conv",
    "XLM-roberta_layers11-15": "XLM-roberta_layers11-15",
}
def add_semantic_results(df, semantic):
    df = df.copy()
    df["corr_concreteness"] = np.nan
    df["corr_abstractness"] = np.nan

    for (binder, semantic_model), arr in semantic.items():
        if semantic_model not in MODEL_MAP:
            continue

        df_model = MODEL_MAP[semantic_model]
        col = (
            "corr_concreteness"
            if binder == "binder_concreteness"
            else "corr_abstractness"
        )

        mask = df["model"] == df_model
        if not mask.any():
            continue

        idx = df.loc[mask, "index"].values
        valid = idx < len(arr)

        df.loc[mask[mask].index[valid], col] = arr[idx[valid]]

    return df



In [34]:
final_df = add_semantic_results(df, semanticresults)

In [35]:
final_df

,index,model,value,run,participant,roi,roi_type,participant_group,corr_concreteness,corr_abstractness
0,0,qwen-text_layers28-32_conv,0.027176,1,33,2,language,blind,0.270019,-0.079389
1,1,qwen-text_layers28-32_conv,0.015785,1,33,2,language,blind,0.068488,0.104302
2,2,qwen-text_layers28-32_conv,0.030376,1,33,2,language,blind,-0.055686,-0.057188
3,3,qwen-text_layers28-32_conv,0.023361,1,33,2,language,blind,0.044549,-0.005386
4,4,qwen-text_layers28-32_conv,0.095281,1,33,2,language,blind,0.137672,0.129640
...,...,...,...,...,...,...,...,...,...,...
5945285,1572,word2vec,-0.071355,6,32,visual,visual,ctrlAV,0.019735,0.005785
5945286,1573,word2vec,0.080193,6,32,visual,visual,ctrlAV,0.039018,0.065027
5945287,1574,word2vec,0.001791,6,32,visual,visual,ctrlAV,0.046878,0.060833
5945288,1575,word2vec,0.013398,6,32,visual,visual,ctrlAV,0.036981,0.052690


In [10]:
nan_rows = df[df[["corr_concreteness", "corr_abstractness"]].isna().any(axis=1)]




KeyError: "None of [Index(['corr_concreteness', 'corr_abstractness'], dtype='object')] are in the [columns]"

In [22]:
df[df[["value"]].isna().any(axis=1)]["model"].unique()

array([], dtype=object)

In [9]:
nan_rows

NameError: name 'nan_rows' is not defined

In [23]:
df.to_csv("results_per_run.csv", sep=";", index=False)